# Self-Reflection & Critique Practical

This notebook builds a **writing critic agent** that improves a draft through a controlled loop:

```text
your draft → critic scores it against a rubric → reflection memory → refiner rewrites → stop or retry
```

Earlier prompting work focused on getting a model to produce useful outputs. This practical adds a second skill: designing a system that can **judge and improve** an output before it is accepted.

**How to use this notebook:** run every cell from top to bottom. The last code cell will ask you to paste a draft. Everything that happens after that — every critic response, every rewrite, and the score on each round — is printed in that one cell's output.

## Three sample drafts to try

If you do not have a draft of your own, copy one of these into the input box in the **last cell** of this notebook.

Each one is weak in a different way, so each one fails a different part of the rubric. Try to guess the score before you run it.

The task they are all answering: *a launch email for working professionals thinking about a live weekend programme in Generative AI and Agentic AI. 140–180 words, warm but professional, mention live classes, hands-on projects and career relevance, no salary promises, one clear call to action.* The full brief is printed further down.

---

**Draft 1 — all hype, no detail.** Watch it fail on specificity and evidence.

```text
AI is changing the world and you should learn it as soon as possible. Our course
will make you skilled in all AI tools and help you get better opportunities. You
will study Generative AI and Agentic AI with experienced trainers and projects.
This is a great chance to upgrade yourself.

Join now to become future ready.
```

---

**Draft 2 — specific, but the numbers are invented.** This one usually scores *higher* on specificity, which is the trap. Watch the constitution catch the made-up figures instead.

```text
Our graduates typically see a 40% salary increase within six months, and 9 out of
10 report better job satisfaction. Ranked the number one AI programme in the
country, this weekend course guarantees placement support and industry-recognised
certification.

Studies show professionals who learn AI now will earn double by 2027. Do not miss
this once-in-a-lifetime opportunity.

Enrol today and transform your future.
```

---

**Draft 3 — accurate and readable, but aimed at the wrong reader.** Nothing here is false. It breaks the audience, length and call-to-action rules instead.

```text
Hey!! 🚀 So basically we're running this SUPER cool weekend thing on GenAI and
agents and it's gonna be lit. You'll learn loads of stuff about transformers,
RAG pipelines, vector embeddings, agentic orchestration frameworks, MCP servers,
and honestly way more than I can list here.

Anyway lmk if you're keen!!
```

---

**What to watch for:** all three come back with a low first score, but for *different reasons*. Read the critic's `issues` list before you look at the rewrite — that list is the actual output of this session.

## Practical architecture

The system has three roles:

1. **Generator** — creates a first draft, or accepts a draft you supply.
2. **Critic** — scores the draft using a rubric and principles.
3. **Refiner** — rewrites the draft using structured critique.

The code owns the stopping rule. The model can recommend whether something passes, but the application enforces the threshold.

In [1]:
from pathlib import Path

from critic_loop import (
    critique_draft,
    print_iteration_trace,
    read_text,
    refine_draft,
    run_refinement_loop,
)
from llm_client import check_api_ready, load_settings

DATA_DIR = Path("data")
PASS_SCORE = 8
MAX_ITERATIONS = 3

settings = load_settings()

print(f"Model: {settings.model}")
print(f"Offline demo mode: {settings.offline_demo}")

# TRAINER NOTE: If the classroom has API/network issues, set BIA_OFFLINE_DEMO=true in .env.
check_api_ready(settings)

Model: gpt-4o-mini
Offline demo mode: False


## The standards live in files. The draft is yours.

The critic needs more than “is this good?” It needs explicit judgment criteria.

- The **brief** defines the target output.
- The **rubric** turns quality into observable criteria, scored 1–10.
- The **constitution** adds principles that prevent unsafe or misleading revisions.

Those three stay on disk. They are the standards, and every run in the room has to be graded against the same bar for two scores to mean anything next to each other.

The **draft** is the one thing you supply, and you will type it in at the bottom of this notebook.

In [2]:
brief = read_text(DATA_DIR / "writing_brief.txt")
rubric = read_text(DATA_DIR / "rubric.md")
constitution = read_text(DATA_DIR / "constitution.md")

print("BRIEF\n")
print(brief)
print("\n" + "-" * 78 + "\n")
print("RUBRIC\n")
print(rubric)
print("\n" + "-" * 78 + "\n")
print("CONSTITUTION\n")
print(constitution)

BRIEF

Write a short launch email for working professionals who are considering a live weekend program in Generative AI and Agentic AI Development.

Audience:
- Software engineers, data analysts, DevOps professionals, and tech managers
- They are busy, skeptical of hype, and want proof that the program is practical

Constraints:
- 140–180 words
- Warm but professional tone
- Mention live expert-led classes, hands-on projects, and career relevance
- Do not make unrealistic salary promises
- End with one clear call to action

------------------------------------------------------------------------------

RUBRIC

# Writing Quality Rubric

Score each draft from 1 to 10 using these criteria:

1. Audience fit — speaks to working professionals without hype.
2. Specificity — includes concrete program benefits, not vague claims.
3. Constraint following — respects word count, tone, and required elements.
4. Evidence quality — avoids unsupported promises or unrealistic outcomes.
5. Call to action

## What the critic returns

The critic does not reply with a paragraph of opinion. It returns structured fields, because that structure is what lets code make decisions:

- **`score`** — a number, so you can set a pass mark and compare one round against the next.
- **`passed`** — a recommendation. The application recomputes it; a blocking issue overrules a high score.
- **`issues`** — each one carries a `criterion`, a `severity` (`minor` / `major` / `blocking`), an `explanation`, and a `revision_instruction`.
- **`reflection_memory`** — one short lesson, carried into the next round.
- **`revised_strategy`** — how the refiner should approach the rewrite.

The important design choice: do not ask the model for a hidden chain of thought. Ask for **fields you can act on**.

`schemas.py` enforces all of this with Pydantic. `revision_instruction` has a minimum length, so “fix it” fails validation, and the issue list is capped at five so the refiner is not handed twenty things at once.

## What the refiner receives

The refiner never gets a vague instruction like “make it better.” It gets:

- the original brief,
- the current draft,
- the structured critique,
- the principles that must stay true,
- the reflection memory from earlier rounds.

It returns the rewritten draft and nothing else — no commentary, no explanation of what it changed.

## The loop, and how it stops

`run_refinement_loop()` repeats critique → refine until one of these fires:

- the score reaches `PASS_SCORE` **and** no blocking issue is present,
- `MAX_ITERATIONS` is reached,
- (in production you would also stop when the score stops improving).

Watch for the case where the score climbs and then flattens — 3 → 7 → 7 is a common result. That plateau means the loop fixed everything that was a wording problem and then ran out of things it could fix without new facts. It is the most useful thing this demo produces, so the output below labels it `NOT PASSED` rather than presenting a 7/10 as a finished draft.

In [ ]:
LINE = "=" * 78


def banner(left: str, right: str = "") -> None:
    """Print a titled separator bar."""
    print("\n" + LINE)
    if right:
        pad = max(1, 78 - len(left) - len(right))
        print(left + " " * pad + right)
    else:
        print(left)
    print(LINE)


def ask_for_draft() -> str:
    """Ask for the draft. One box, one Enter, done."""
    print("Paste your draft in the box below, then press Enter.")
    print(LINE)
    try:
        return input("your draft > ").strip()
    except (EOFError, KeyboardInterrupt):
        return ""


def show_run(draft: str, final_draft: str, trace: list, pass_score: int) -> None:
    """Print the whole flow: a short summary first, then every step in full."""
    # ---- the story in a few lines, so it survives a truncated output pane ----
    banner("THE RUN AT A GLANCE", f"{len(trace)} rounds")
    step = 1
    print(f"  step {step}  your draft{'':<24}{len(draft.split())} words")
    previous = None
    for index, record in enumerate(trace):
        step += 1
        change = "" if previous is None else f"  ({record.score - previous:+d})"
        mark = "PASSED" if record.passed else "too low"
        print(f"  step {step}  critic scores it{'':<17}{record.score}/10  {mark}{change}")
        previous = record.score
        has_rewrite = index + 1 < len(trace) or not record.passed
        if has_rewrite:
            step += 1
            print(f"  step {step}  refiner rewrites it")

    last = trace[-1]
    best = max(record.score for record in trace)
    if last.passed:
        print(f"\n  RESULT: PASSED - {last.score}/10, pass mark {pass_score}/10")
    else:
        print(f"\n  RESULT: NOT PASSED - best {best}/10, pass mark {pass_score}/10")
    print("\n  (full detail below - if your output pane cuts off, click")
    print("   'View as a scrollable element' underneath it)")

    # ---- the same thing again, in full ----
    step = 1
    banner(f"STEP {step} - YOUR DRAFT", f"{len(draft.split())} words")
    print(draft)

    previous = None
    for index, record in enumerate(trace):
        critique = record.critique

        step += 1
        title = "THE CRITIC SCORES YOUR DRAFT" if previous is None else "THE CRITIC SCORES THE REWRITE"
        verdict = "PASSED" if critique.passed else f"below the pass mark of {pass_score}"
        change = "" if previous is None else f"  ({critique.score - previous:+d})"
        banner(f"STEP {step} - {title}", f"{critique.score}/10  {verdict}{change}")

        print(critique.summary)
        if critique.issues:
            print(f"\nWhat it wants changed ({len(critique.issues)}):")
            for number, issue in enumerate(critique.issues, start=1):
                print(f"  {number}. [{issue.severity}] {issue.criterion}")
                print(f"     {issue.revision_instruction}")
        print(f"\nLesson carried to the next round: {critique.reflection_memory}")
        previous = critique.score

        # This round's rewrite is the next round's draft. After the last round
        # it is whatever the loop returned.
        if index + 1 < len(trace):
            rewrite = trace[index + 1].draft
        elif not critique.passed:
            rewrite = final_draft
        else:
            rewrite = None

        if rewrite is not None:
            step += 1
            banner(f"STEP {step} - THE REFINER REWRITES IT",
                   f"applying those {len(critique.issues)} instructions")
            print(rewrite)

    banner("HOW THE SCORE MOVED")
    for record in trace:
        print(f"  round {record.iteration}   {record.score:>2}/10   {'#' * record.score}")

    scores = " -> ".join(str(record.score) for record in trace)

    if last.passed:
        banner(f"RESULT: PASSED - {last.score}/10 on round {last.iteration}",
               f"round scores {scores}")
        print("\nFINAL DRAFT\n")
    else:
        banner(f"RESULT: NOT PASSED - best {best}/10, needed {pass_score}/10",
               f"round scores {scores}")
        print(f"\nStopped after {len(trace)} rounds without reaching the pass mark.")
        print("The text below is the rewrite made after the last critique.")
        print("It was never scored - the retry budget ran out first.\n")
        print("DRAFT AS IT STANDS - NOT PASSED\n")

    print(final_draft)


print("Helpers ready.")

## Bridge to the travel planner project

The same pattern can critique itinerary quality. Later you swap the writing brief for the bridge brief and write a travel-specific rubric: budget realism, feasible timing, personalization, logical ordering, clear trade-offs.

The critic, the schema and the stopping rules stay exactly as they are.

In [4]:
print(read_text(DATA_DIR / "travel_planner_bridge_brief.txt"))

Upcoming project bridge:

The same evaluator-generator loop can critique a travel itinerary.

Example itinerary criteria:
- Budget realism
- Logical ordering of activities
- Time feasibility
- Personalization to user preferences
- Clear explanation of trade-offs


---

# Run it

The next cell is the whole practical.

It will ask you for a draft. Paste one in and **press Enter once** — that is all. One box, one Enter.

If you have nothing to hand, scroll back to **Three sample drafts to try** at the top of this notebook and copy one of those.

You will then see the whole flow, step by step:

| | |
|---|---|
| **Step 1** | your draft, exactly as you typed it |
| **Step 2** | the critic scores it and lists what it wants changed |
| **Step 3** | the refiner rewrites it using those instructions |
| **Step 4** | the critic scores the rewrite, and you see the score move |
| … | repeating until it passes or runs out of rounds |

Then a summary of how the score moved, the verdict, and the final draft.

**Read step 2 before you read step 3.** The critic's list is the real output of this session — the rewrite is just the consequence of it.

> **If the output looks like it stops halfway,** it is not broken — VS Code trims notebook output to 30 lines by default. Click **View as a scrollable element** underneath the output, or raise `notebook.output.textLineLimit` in VS Code settings to something like 2000. The run always prints a short summary at the top so you can see the whole story even when the rest is trimmed.

In [ ]:
# COST NOTE: 1 critic call per round, plus 1 refiner call after each failed round.
# With MAX_ITERATIONS = 3 that is at most 6 short calls.

student_draft = ask_for_draft()

if not student_draft:
    raise ValueError("Nothing was entered, so nothing was sent to the model. Run this cell again.")

print(f"\nGot {len(student_draft.split())} words. Sending it to the critic...")

final_draft, trace = run_refinement_loop(
    brief=brief,
    initial_draft=student_draft,
    rubric=rubric,
    constitution=constitution,
    pass_score=PASS_SCORE,
    max_iterations=MAX_ITERATIONS,
)

show_run(student_draft, final_draft, trace, PASS_SCORE)

---

## Challenge 1 — Make the critic stricter

Change `PASS_SCORE` to `9` in the setup cell and run the last cell again with the same draft.

- Does the loop take more rounds?
- Does the final draft get better, or just longer?
- At what point does more refinement stop helping?

## Challenge 2 — Take the rubric away

Pass an empty string as the `rubric` and run the same draft through. The critic still returns a score. Ask the class what that score is now measuring.

## Optional — AutoGen evaluator preview

AutoGen appears here only as a preview; the framework itself comes later. The point is that the evaluator role can be wrapped as an agent while the pattern underneath stays identical.

```bash
pip install -r requirements_autogen.txt
```

```python
from autogen_evaluator_preview import run_autogen_evaluator_preview

autogen_result = await run_autogen_evaluator_preview(
    brief=brief,
    draft=student_draft,
    rubric=rubric,
    constitution=constitution,
)
print(autogen_result)
```

## Wrap-up

You built a practical reflection loop with:

- rubric-based critique,
- structured evaluator output,
- principles-based correction,
- reflection memory,
- threshold-based stopping,
- an honest verdict when the loop does not pass.

The key lesson: self-reflection is useful only when it is designed as a controlled system, not as a loose prompt asking the model to “think again.”